# Robot arm, live testing 24.02.2026

### Connect to the simulator / to the robot

In [1]:
import sys
from pathlib import Path
from math import cos, sin, pi, radians

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)

from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor


# # # Connect to the simulator
# iscoin = ISCoin()
# robot = iscoin.robot_control
# print("Connected to simulator")

In [2]:
from URBasic import ISCoin

# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

2026-02-24 14:59:10,111,111 | ISCoin - 10.30.5.158 | DEBUG    [iscoin.py:59] Initializing devices in constructor
2026-02-24 14:59:10,112,112 | ISCoin - 10.30.5.158 | INFO     [iscoin.py:106] Reinitializing devices with IP 10.30.5.158
 * Robot model
2026-02-24 14:59:10,113,113 | ISCoin - 10.30.5.158 | INFO     [iscoin.py:108]  * URScriptExt
2026-02-24 14:59:11,116,116 | ISCoin - 10.30.5.158 | INFO     [iscoin.py:110]  * RobotiqTwoFingersGripper
2026-02-24 14:59:11,117,117 | ISCoin - 10.30.5.158 | INFO     [iscoin.py:112]  * RobotiqWristCamera
2026-02-24 14:59:11,118,118 | ISCoin - 10.30.5.158 | INFO     [iscoin.py:114]  *** Devices initialized


In [3]:
robot = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [21]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot.movej(home, a=radians(80), v=radians(60))

tcp_home = robot.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

Home TCP: TCP6D([-0.003831243051278224, -0.34865872671954806, 0.10531107795346409, -0.011627697102678893, -3.135867485420041, -0.009315519477994505])


In [ ]:
# target_tcp = TCP6D.createFromMetersRadians(
#       0.0,      # x
#      -0.35,     # y
#       0.20,     # z
#       0.0,      # rx
#       3.14,      # ry
#       0.0       # rz
#   )

In [ ]:
# current_joints = robot.get_actual_joint_positions()
# target_joints = robot.get_inverse_kin(target_tcp, qnear=current_joints)

## Set TCP

In [11]:
pen_length = 0.2385  # meters
robot.set_tcp(TCP6D.createFromMetersRadians(0, 0, pen_length, 0, 0, 0))

In [12]:
tcp = robot.get_actual_tcp_pose()
print(tcp)

TCP6D([-0.0038555002984891735, -0.3486367720221684, 0.10535109121838818, -0.011547364713884557, -3.1358152770757712, -0.009218186039550066])


## Verify the Z plan for 2d drawing

In [ ]:
robot.reset_error()

In [ ]:
# Enable — you can physically move the arm by hand
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p1 = robot.get_actual_tcp_pose()
print(p1)

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p2 = robot.get_actual_tcp_pose()
print(p2)

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p3 = robot.get_actual_tcp_pose()
print(p3)

Verification of z values

In [ ]:
print(f"Z values: {p1.z:.4f}, {p2.z:.4f}, {p3.z:.4f}")
print(f"Z spread: {max(p1.z, p2.z, p3.z) - min(p1.z, p2.z, p3.z):.4f} m")

# Gripper setup ( manually )

In [23]:
iscoin.gripper.activate()

True

In [76]:
iscoin.gripper.deactivate()

True

In [ ]:
from time import sleep
if not iscoin.gripper.open():
  print("Failed to open gripper")
else:
  sleep(1)
  if not iscoin.gripper.close():
    print("Failed to close gripper")


In [75]:
iscoin.gripper.open()

True

In [77]:
iscoin.gripper.close()

False

### Drawing algorithm live

Drawings parameters

In [67]:
drawing_z = 0.0014
safe_z = drawing_z + 0.03

rx, ry, rz = 0.0, 3.14, 0.0

draw_speed = 0.05   # m/s — slow for drawing
travel_speed = 0.1   # m/s — faster for travel moves

After testing: drawing_z = 0.0015 is good

In [6]:
sim_ref_point = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.02,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )

In [7]:
robot.movel(sim_ref_point, v=travel_speed)

movej sent (duration=2s)
Movement OK — target reached


True

Set drawing reference point

In [14]:
robot.freedrive_mode()

In [15]:
robot.end_freedrive_mode()

# Take x, y from where you placed the arm, z and orientation from drawing parameters
ref = robot.get_actual_tcp_pose()
ref_drawing_point = TCP6D.createFromMetersRadians(ref.x, ref.y, drawing_z, rx, ry, rz)
print(f"Drawing ref: x={ref.x:.4f}, y={ref.y:.4f}, z={drawing_z}")

Drawing ref: x=-0.0275, y=-0.3784, z=0.005


In [13]:
#### FOR SIMULATION

ref_drawing_point = TCP6D.createFromMetersRadians(0, -0.35, drawing_z, rx, ry, rz)

Generate trajectory

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [47]:
# Circle: radius 5cm, 32 segments
r = 0.05
n = 32

path_offsets = [(r * cos(2 * pi * i / n), r * sin(2 * pi * i / n)) for i in range(n + 1)]
print(f"{len(path_offsets)} offsets generated")

33 offsets generated


In [68]:
# Lemniscate of Bernoulli: width 10cm, 64 segments
a = 0.05
n = 64

path_offsets = []
for i in range(n + 1):
    t = 2 * pi * i / n
    denom = 1 + sin(t) ** 2
    dx = a * cos(t) / denom
    dy = a * sin(t) * cos(t) / denom
    path_offsets.append((dx, dy))

print(f"{len(path_offsets)} offsets generated (lemniscate)")

65 offsets generated (lemniscate)


In [11]:
# Square: side 10cm, 8 subdivisions per side
side = 0.10
n_per_side = 8
half = side / 2

corners = [(-half, -half), (half, -half), (half, half), (-half, half), (-half, -half)]

path_offsets = []
for j in range(len(corners) - 1):
    x0, y0 = corners[j]
    x1, y1 = corners[j + 1]
    for i in range(n_per_side):
        t = i / n_per_side
        path_offsets.append((x0 + t * (x1 - x0), y0 + t * (y1 - y0)))
path_offsets.append(corners[-1])

print(f"{len(path_offsets)} offsets generated (square)")

33 offsets generated (square)


Convert to TCP waypoints

In [69]:
cx, cy = ref_drawing_point.x, ref_drawing_point.y

waypoints = []
for i, (dx, dy) in enumerate(path_offsets):
    tcp = TCP6D.createFromMetersRadians(cx + dx, cy + dy, drawing_z, rx, ry, rz)
    blend = 0.005 if i < len(path_offsets) - 1 else 0.0
    waypoints.append(TCP6DDescriptor(tcp, v=draw_speed, r=blend))

print(f"{len(waypoints)} waypoints ready")

65 waypoints ready


Move to start above

In [70]:
start_above = TCP6D.createFromMetersRadians(cx + path_offsets[0][0], cy + path_offsets[0][1], safe_z, rx, ry, rz)
robot.movel(start_above, v=travel_speed)

Move down to paper (A)

In [71]:
start_down = TCP6D.createFromMetersRadians(cx + path_offsets[0][0], cy + path_offsets[0][1], drawing_z, rx, ry, rz)
robot.movel(start_down, v=travel_speed)

Draw

In [72]:
for i in range(3):
    robot.movel_waypoints(waypoints)

Lift up and go home

In [73]:
end_above = TCP6D.createFromMetersRadians(cx + path_offsets[-1][0], cy + path_offsets[-1][1], safe_z, rx, ry, rz)
robot.movel(end_above, v=travel_speed)

In [74]:
robot.movej(home, a=radians(80), v=radians(60))